## import libraries

In [1]:
platformID = 'YT-'

In [2]:
from IPython.display import display

import os
import zipfile

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.max_colwidth', None)

import numpy as np

import re

#import yxdb

import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns 

import psycopg2

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader, execute_sql_query
import test_functions

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes'])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test YT-_1c_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_02 passed: No empty values in lookup.
...updating logbook...

✅ Test YT-_1c_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_05 passed: No empty values in lookup.
...updating logbook...

✅ Test YT-_1c_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_08 passed: No empty values in lookup.
...updating logbook...



# automated 

In [5]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

sql_query = f"""
    SELECT
        week_commencing,
        channel_id,
        channel_name,
        country_code,
        views_country
    FROM
        central_insights.adverity_social_youtube_by_channel_geo
    WHERE
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['channel_id'] = platformID+df['channel_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

yt_views_raw = pd.read_csv(file, keep_default_na=False)
yt_views_raw['week_commencing'] = pd.to_datetime(yt_views_raw['week_commencing'])
yt_views_raw['country_code'] = yt_views_raw['country_code'].replace('', 'ZZ')
yt_views_raw = yt_views_raw.rename(columns = {'week_commencing': 'w/c',
                                              'channel_id': 'Channel ID',
                                              'channel_name': 'Channel Name',
                                              'country_code': 'YT-_FBE_codes'})

no redshift connection using last time queried


In [6]:
yt_views_raw = yt_views_raw[yt_views_raw['Channel ID'].isin(channel_ids)]

### RUN TESTS


# missing page_ids
test_functions.test_filter_elements_returned(yt_views_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=yt_views_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(yt_views_raw, 
                           numeric_columns=['views_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(yt_views_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test YT-_1c_09 passed: everything found!
...updating logbook...

✅ No missing weeks from each channel’s Start onward.
...updating logbook...

✅ Test YT-_1c_11 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test YT-_1c_12 passed: No combinations occurs more than once a week.
...updating logbook...



In [7]:
# add PlaceID
cols = ['PlaceID', 'YT-_FBE_codes']
yt_views = yt_views_raw.merge(country_codes[cols],
                              on=['YT-_FBE_codes'],
                              how='left').drop(columns=['YT-_FBE_codes'])
test_functions.test_inner_join(yt_views_raw, 
                               country_codes[cols], 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='adding country codes GAM', 
                               focus='left')



✅ Inner join test YT-_1c_13 successful: No issues found.
...updating logbook...



In [8]:
# Group by the specified columns and sum the yt_metric_value
yt_views_global = yt_views.groupby([ 'Channel ID','w/c']).agg({'views_country': 'sum'}).reset_index()
yt_views_global = yt_views_global.rename(columns={'views_country': 'total_views_country'})
#display(grouped_df_allCountries.sample())

yt_country = yt_views.merge(yt_views_global,
                                    on=['Channel ID', 'w/c'],
                                    how='inner')
test_functions.test_inner_join(yt_views, 
                               yt_views_global, 
                               ['Channel ID', 'w/c'],
                               f"{platformID}_1c_14",
                               test_step='combining country and global views', 
                               focus='both')


yt_country['country_%'] = (yt_country['views_country'] / yt_country['total_views_country'])

test_functions.test_percentage(yt_country,  
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15",
                               test_step = 'calculating % country')
yt_country

✅ Inner join test YT-_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



,w/c,Channel ID,Channel Name,views_country,PlaceID,total_views_country,country_%
0,2025-11-17,YT-UC3780MVtSV3Huj4si_CQvlQ,BBC News සිංහල,4302,USA,490136,0.008777
1,2025-11-17,YT-UC8zGv2QY2-l6dyiyF9H2wzA,BBC News Кыргыз,481,TUR,555003,0.000867
2,2025-11-17,YT-UCBte7YLdJx-O_YljuvN6whg,BBC Afrique,28,IRN,139793,0.000200
3,2025-11-17,YT-UCF4QKhPMP1JybbkIJzIayww,BBC News India,302,EGY,674005,0.000448
4,2025-11-17,YT-UCHZk9MrT3DGWmVqdsj5y0EA,BBC Persian,2418,GRE,2853657,0.000847
...,...,...,...,...,...,...,...
173987,2026-01-12,YT-UCUBIrDsIVzRpKsClMlSlTpQ,BBC News Mundo,21419,FRA,3210851,0.006671
173988,2026-01-12,YT-UCUBIrDsIVzRpKsClMlSlTpQ,BBC News Mundo,19702,UK,3210851,0.006136
173989,2026-01-12,YT-UCUBIrDsIVzRpKsClMlSlTpQ,BBC News Mundo,617,EQU,3210851,0.000192
173990,2026-01-12,YT-UCUBIrDsIVzRpKsClMlSlTpQ,BBC News Mundo,666,GRE,3210851,0.000207


# manual

# store

In [9]:

file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
yt_country[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_REDSHIFT_COUNTRY.csv", 
                         index=None, na_rep='')

In [10]:
yt_country['w/c'].max()

Timestamp('2026-01-12 00:00:00')